# Semana 05

## Importações e Configuração Inicial
Vamos começar por importar as bibliotecas de criptografia e definir as funções base de padding e unpadding fornecidas no guião.

In [1]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding

BLOCK_SIZE_BITS = 128
BLOCK_SIZE_BYTES = BLOCK_SIZE_BITS // 8

def pad(ptxt):
    padder = padding.PKCS7(BLOCK_SIZE_BITS).padder()
    padded = padder.update(ptxt) + padder.finalize()
    return padded

def unpad(padded):
    unpadder = padding.PKCS7(BLOCK_SIZE_BITS).unpadder()
    ptxt = unpadder.update(padded) + unpadder.finalize()
    return ptxt

def ints(bstr):
    return [int(x) for x in bstr]

# Teste básico
print("Padding de 'abc':", ints(pad(b"abc")))

Padding de 'abc': [97, 98, 99, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13]


### Questão 01
**Objetivo:** Ilustrar um erro de unpadding onde o tamanho está correto (múltiplo do tamanho do bloco), mas o formato do padding é inválido.

In [4]:
mensagem = b"abc"
mensagem_com_padding = bytearray(pad(mensagem))

print(f"Tamanho da mensagem com padding: {len(mensagem_com_padding)} bytes (Múltiplo de {BLOCK_SIZE_BITS//8} bytes)")
print(f"Último byte original (tamanho do padding): {mensagem_com_padding[-1]}")

# corromper o formato do padding (mantendo o tamanho total da mensagem intocável)
# alterar o último byte para um valor inválido, por exemplo 0x99 (153 em decimal)
mensagem_com_padding[-1] = 0x99 

# tentar remover o padding
try:
    unpad(bytes(mensagem_com_padding))
    print("Sucesso inesperado!")
except ValueError as e:
    print(f"\nErro de Unpadding capturado com sucesso!")
    print(f"Detalhe do erro: {e}")

Tamanho da mensagem com padding: 16 bytes (Múltiplo de 16 bytes)
Último byte original (tamanho do padding): 13

Erro de Unpadding capturado com sucesso!
Detalhe do erro: Invalid padding bytes.


**Explicação:** O tamanho da mensagem continua a ser múltiplo do bloco, mas o valor do padding (0x99) é inválido.

## Preparação da Cifra (AES/CBC)
Antes de construir o padding oracle, precisamos de ter um mecanismo real de cifra e decifra para emular o servidor/sistema atacado.

In [5]:
# Gerar chaves e IV aleatórios para a sessão
KEY = os.urandom(BLOCK_SIZE_BYTES)

def enc(ptxt):
    """
    Cifra o texto limpo usando AES/CBC e retorna IV + Criptograma.
    """
    iv = os.urandom(BLOCK_SIZE_BYTES)
    cipher = Cipher(algorithms.AES(KEY), modes.CBC(iv))
    encryptor = cipher.encryptor()
    padded_ptxt = pad(ptxt)
    ctxt = encryptor.update(padded_ptxt) + encryptor.finalize()
    return iv + ctxt

def dec(iv_and_ctxt):
    """
    Decifra o criptograma (incluindo o IV inicial) e retorna o texto com padding.
    """
    iv = iv_and_ctxt[:BLOCK_SIZE_BYTES]
    ctxt = iv_and_ctxt[BLOCK_SIZE_BYTES:]
    cipher = Cipher(algorithms.AES(KEY), modes.CBC(iv))
    decryptor = cipher.decryptor()
    padded_ptxt = decryptor.update(ctxt) + decryptor.finalize()
    return padded_ptxt

## PROG: **`cbc_pad_orcl.py`**
**Objetivo:** Instanciar e testar o Padding Oracle.

In [6]:
def pad_orcl(iv_and_ctxt):
    padded_msg = dec(iv_and_ctxt)
    unpadder = padding.PKCS7(BLOCK_SIZE_BITS).unpadder()
    try:
        unpadder.update(padded_msg) + unpadder.finalize()
        return True
    except ValueError:
        return False

**Teste:**

In [7]:
mensagem_teste = input("Escreve a mensagem para cifrar: ").encode()

criptograma_valido = enc(mensagem_teste)

print("\nCriptograma válido:", criptograma_valido.hex())
print("Oráculo com criptograma válido:", pad_orcl(criptograma_valido))

# criar criptograma malformado
criptograma_malformado = bytearray(criptograma_valido)
criptograma_malformado[-1] ^= 0xFF
criptograma_malformado = bytes(criptograma_malformado)

print("\nCriptograma malformado:", criptograma_malformado.hex())
print("Oráculo com criptograma malformado:", pad_orcl(criptograma_malformado))

Escreve a mensagem para cifrar:  ola



Criptograma válido: ff068fd91bfacd8671057317bc7370ec49b65d2cd708fd8a60243ff69d422a2f
Oráculo com criptograma válido: True

Criptograma malformado: ff068fd91bfacd8671057317bc7370ec49b65d2cd708fd8a60243ff69d422ad0
Oráculo com criptograma malformado: False


**Explicação:** A função que tenta decifrar um criptograma e validar o seu *padding*, devolvendo apenas um `True` (sucesso) ou `False` (erro de formato). É esta resposta binária ("sim/não") que permite ao atacante deduzir o texto limpo, atuando como um "oráculo".

## PROG: **`pad_orcl_attck_lastbyte.py`**
**Objetivo:** Descobrir o último byte (e consequentemente o tamanho do padding) de um bloco.

In [23]:
def descobrir_tamanho_padding(iv_and_ctxt):
    """
    Ataca o último byte manipulando o penúltimo bloco do criptograma
    para descobrir o tamanho do padding utilizado.
    """
    # dividir o criptograma em blocos de 16 bytes
    blocos = [iv_and_ctxt[i:i+BLOCK_SIZE_BYTES] for i in range(0, len(iv_and_ctxt), BLOCK_SIZE_BYTES)]
    
    if len(blocos) < 2:
        raise ValueError("Criptograma demasiado curto.")
        
    c_prev = bytearray(blocos[-2])
    c_last = blocos[-1]            
    
    # iterar os 256 valores possíveis (0x00 a 0xFF) para adivinhar o último byte
    for guess in range(256):
        c_modificado = bytearray(c_prev)
        
        # fórmula Mágica do Padding Oracle:
        # C_modificado = C_original XOR Tentativa XOR Padding_Desejado (0x01)
        c_modificado[-1] = c_prev[-1] ^ guess ^ 0x01
        
        # testar no oráculo
        if pad_orcl(bytes(c_modificado) + c_last):
            
            c_check = bytearray(c_modificado)
            c_check[-2] ^= 0xFF # inverter bits do penúltimo byte
            
            # se continuar a ser válido após corromper o penúltimo byte, 
            # é porque o padding é garantidamente apenas "0x01"!
            if pad_orcl(bytes(c_check) + c_last):
                return guess # encontrámos o byte. No PKCS7, este valor é o tamanho do padding.

**Teste:**

In [24]:
tamanho_pad = descobrir_tamanho_padding(criptograma_valido)
print(f"O tamanho do padding neste criptograma é: {tamanho_pad} bytes")

O tamanho do padding neste criptograma é: 13 bytes


**Explicação:** Manipula o penúltimo bloco para alterar o comportamento do último bloco durante a decifragem. Testa 256 valores diferentes (`0x00` a `0xFF`) até o oráculo devolver `True`, o que indica que conseguimos forçar o último byte a ser `0x01`. Com uma operação matemática (XOR), isto revela o valor real do byte (tamanho do padding). Inclui uma proteção contra falsos positivos.

## PROG: **`pad_orcl_attck.py`**
**Objetivo:** Recuperar o último bloco inteiro iterando do último byte até ao primeiro.

In [25]:
def recuperar_ultimo_bloco(iv_and_ctxt):
    """
    Usa o padding oracle attack para recuperar todo o último bloco de texto limpo.
    """
    blocos = [iv_and_ctxt[i:i+BLOCK_SIZE_BYTES] for i in range(0, len(iv_and_ctxt), BLOCK_SIZE_BYTES)]
    
    if len(blocos) < 2:
        raise ValueError("Criptograma demasiado curto.")
        
    c_prev = bytearray(blocos[-2])
    c_last = blocos[-1]            
    
    texto_limpo = bytearray(BLOCK_SIZE_BYTES)
    
    # vamos iterar de trás para a frente (do byte 15 até ao byte 0)
    for i in range(BLOCK_SIZE_BYTES - 1, -1, -1):
        padding_val = BLOCK_SIZE_BYTES - i
        byte_encontrado = False
        
        for guess in range(256):
            c_modificado = bytearray(c_prev)
            
            # ajustar os bytes que já descobrimos (à direita de 'i')
            for k in range(i + 1, BLOCK_SIZE_BYTES):
                c_modificado[k] = c_prev[k] ^ texto_limpo[k] ^ padding_val
                
            # forçar a nossa tentativa no byte atual 'i'
            c_modificado[i] = c_prev[i] ^ guess ^ padding_val
            
            # testar no oráculo
            if pad_orcl(bytes(c_modificado) + c_last):
                # se estamos no último byte, garantimos que o padding é APENAS de tamanho 1
                if i == BLOCK_SIZE_BYTES - 1:
                    c_check = bytearray(c_modificado)
                    # corrompemos o penúltimo byte alterando bits ao calhas (XOR 0xFF)
                    c_check[i - 1] ^= 0xFF 
                    if not pad_orcl(bytes(c_check) + c_last):
                        # padding quebrou, logo não era 0x01. Falso positivo!
                        continue 
            
                texto_limpo[i] = guess
                byte_encontrado = True
                break # se encontrámos o byte real, saltamos para o próximo 'i'

    return bytes(texto_limpo)

**Teste:**

In [26]:
bloco_recuperado = recuperar_ultimo_bloco(criptograma_valido)
print("Bloco recuperado:", bloco_recuperado)
print("Bloco sem padding:", unpad(bloco_recuperado))

Bloco recuperado: b'ola\r\r\r\r\r\r\r\r\r\r\r\r\r'
Bloco sem padding: b'ola'


**Explicação:** Pega na lógica do ataque de 1 byte e automatiza-a de trás para a frente (do byte 15 ao byte 0). Usa os bytes que já descobriu para forçar os paddings seguintes (`0x02` `0x02`, `0x03` `0x03` `0x03`, etc.), quebrando os 16 bytes do bloco inteiro, um a um.